In [1]:
import mlflow
from mlflow.tracking import MlflowClient

In [2]:
import os
os.environ["DATABRICKS_CONFIG_FILE"] = "../secrets/.databrickscfg"

mlflow.login(interactive=False)
print("MLflow login successful.")

2026/03/26 15:20:18 INFO mlflow.utils.credentials: Successfully connected to MLflow hosted tracking server! Host: https://dbc-d0177b72-9a40.cloud.databricks.com.


MLflow login successful.


In [3]:
# mlflow.set_experiment("/Users/harish.akula096@gmail.com/regressors")

In [4]:
client = MlflowClient()

exp = client.get_experiment_by_name("/Users/harish.akula096@gmail.com/regressors")
print(exp.experiment_id)

1279701826324544


In [5]:
import mlflow
import json
from mlflow.tracking import MlflowClient

def load_model_artifacts(model_uuid, dst_path):

    client = MlflowClient()

    exp = client.get_experiment_by_name(
        "/Users/harish.akula096@gmail.com/regressors"
    )
    experiment_id = exp.experiment_id

    runs = mlflow.search_runs(
        experiment_ids=[experiment_id],
        filter_string=f"tags.mlflow.runName = '{model_uuid}'"
    )

    run_id = runs.iloc[0]["run_id"]

    weights_path = client.download_artifacts(
        run_id,
        f"{model_uuid}.best_weights.pt",
        dst_path=dst_path
    )

    config_path = client.download_artifacts(
        run_id,
        "config.json",
        dst_path=dst_path
    )

    with open(config_path) as f:
        config = json.load(f)

    return weights_path, config

weights_path, config = load_model_artifacts("y25y26m1", "./artifacts")

/home/akula.ha/.conda/envs/myenv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
weights_path

'/projects/talisman/akulaha/qqzz4l-NN/notebooks/artifacts/y25y26m1.best_weights.pt'

In [ ]:

def evaluate_model(model, dataloader, device, output_dir, config):
    """
    Evaluate a trained model and save:
      - metrics.json (scaled & unscaled)
      - hist_unscaled_overlay.png, hist_scaled_overlay.png
    Requires a globally available scalar y_unscale(y: float) -> float.
    """
    os.makedirs(output_dir, exist_ok=True)

    model.eval()
    all_preds, all_targets = [], []
    with torch.no_grad():
        for xb, yb in dataloader:
            xb = xb.to(device)
            pred = model(xb).cpu()
            all_preds.append(pred)
            all_targets.append(yb.cpu())

    y_true = torch.cat(all_targets, dim=0).numpy()
    y_pred = torch.cat(all_preds, dim=0).numpy()

    y_names = config["var_y"]
    assert y_true.ndim == 2 and y_true.shape[1] == len(y_names), \
        "Mismatch between targets array shape and config['var_y']"

    # -------- Scaled metrics --------
    scaled = {}
    overall_mape = mean_absolute_percentage_error(y_true, y_pred)
    overall_r2   = r2_score(y_true, y_pred)
    overall_abs  = (1 - overall_mape) * 100 if np.isfinite(overall_mape) else np.nan
    scaled["overall"] = {"MAPE": float(overall_mape), "R2": float(overall_r2), "abs_score": float(overall_abs)}
    scaled["per_target"] = {}
    for j, name in enumerate(y_names):
        mape_j = mean_absolute_percentage_error(y_true[:, j], y_pred[:, j])
        r2_j   = r2_score(y_true[:, j], y_pred[:, j])
        abs_j  = (1 - mape_j) * 100 if np.isfinite(mape_j) else np.nan
        scaled["per_target"][name] = {"MAPE": float(mape_j), "R2": float(r2_j), "abs_score": float(abs_j)}

    # -------- Unscaled metrics --------
    # Your y_unscale is scalar-only; vectorize it.
    y_unscale_vec = np.vectorize(y_unscale)
    y_true_un = y_unscale_vec(y_true)
    y_pred_un = y_unscale_vec(y_pred)

    unscaled = {}
    overall_mape_un = mean_absolute_percentage_error(y_true_un, y_pred_un)
    overall_r2_un   = r2_score(y_true_un, y_pred_un)
    overall_abs_un  = (1 - overall_mape_un) * 100 if np.isfinite(overall_mape_un) else np.nan
    unscaled["overall"] = {"MAPE": float(overall_mape_un), "R2": float(overall_r2_un), "abs_score": float(overall_abs_un)}
    unscaled["per_target"] = {}
    for j, name in enumerate(y_names):
        mape_j_un = mean_absolute_percentage_error(y_true_un[:, j], y_pred_un[:, j])
        r2_j_un   = r2_score(y_true_un[:, j], y_pred_un[:, j])
        abs_j_un  = (1 - mape_j_un) * 100 if np.isfinite(mape_j_un) else np.nan
        unscaled["per_target"][name] = {"MAPE": float(mape_j_un), "R2": float(r2_j_un), "abs_score": float(abs_j_un)}

    # -------- Deltas (arrays, no DataFrame) --------
    scaled_delta   = _percent_delta(y_true,    y_pred)
    unscaled_delta = _percent_delta(y_true_un, y_pred_un)

    # -------- Save metrics JSON --------
    metrics = {"scaled": scaled, "unscaled": unscaled, "meta": {"num_samples": int(y_true.shape[0]), "targets": y_names}}
    out_path = os.path.join(output_dir, "metrics.json")
    with open(out_path, "w") as f:
        json.dump(metrics, f, indent=2)

    # -------- Save plots (overlay style) --------
    _save_error_plots_from_arrays(
        unscaled_delta, y_names, os.path.join(output_dir, "hist_unscaled_overlay.png"), cutoff=5
    )
    _save_error_plots_from_arrays(
        scaled_delta, y_names, os.path.join(output_dir, "hist_scaled_overlay.png"), cutoff=5
    )

    # Console summary
    print("\n=== Evaluation (SCALED) ===")
    print(f"Overall MAPE      : {scaled['overall']['MAPE']:.6f}")
    print(f"Overall R2        : {scaled['overall']['R2']:.6f}")
    print(f"Overall abs_score : {scaled['overall']['abs_score']:.6f}")
    for n, v in scaled["per_target"].items():
        print(f"{n} - MAPE: {v['MAPE']:.6f} | R2: {v['R2']:.6f} | abs_score: {v['abs_score']:.6f}")

    print("\n=== Evaluation (UNSCALED) ===")
    print(f"Overall MAPE      : {unscaled['overall']['MAPE']:.6f}")
    print(f"Overall R2        : {unscaled['overall']['R2']:.6f}")
    print(f"Overall abs_score : {unscaled['overall']['abs_score']:.6f}")
    for n, v in unscaled["per_target"].items():
        print(f"{n} - MAPE: {v['MAPE']:.6f} | R2: {v['R2']:.6f} | abs_score: {v['abs_score']:.6f}")

    print(f"\nSaved metrics to: {out_path}")
    print("Saved histograms to:",
          os.path.join(output_dir, "hist_unscaled_overlay.png"), "and",
          os.path.join(output_dir, "hist_scaled_overlay.png"))
    
    plot_losses_from_output_dir(output_dir)

    return {
        "metrics": metrics,
        "paths": {
            "metrics_json": out_path,
            "hist_unscaled": os.path.join(output_dir, "hist_unscaled_overlay.png"),
            "hist_scaled": os.path.join(output_dir, "hist_scaled_overlay.png"),
            "loss_plot": os.path.join(output_dir, "loss_plot.png"),
            "loss_plot_loglog": os.path.join(output_dir, "loss_plot_loglog.png"),
        }
    }

def flatten_dict(d, parent_key="", sep="."):
    out = {}
    for k, v in d.items():
        key = f"{parent_key}{sep}{k}" if parent_key else k
        if isinstance(v, dict):
            out.update(flatten_dict(v, key, sep=sep))
        else:
            out[key] = v
    return out

def log_to_mlflow(config, uuid_str, output_dir, model, eval_out, input_example=None):

    # params: model param count + flattened config
    mlflow.log_param("num_parameters", int(sum(p.numel() for p in model.parameters())))

    flat = flatten_dict(config)
    for k, v in flat.items():
        mlflow.log_param(k, v)

    # artifacts: config json
    mlflow.log_artifact(os.path.join(output_dir, "config.json"))

    # # metrics: scaled & unscaled (overall + per-target)
    metrics = eval_out["metrics"]
    # for scale in ["scaled", "unscaled"]:
    #     mlflow.log_metric(f"{scale}_overall_mape", metrics[scale]["overall"]["MAPE"])
    #     mlflow.log_metric(f"{scale}_overall_r2", metrics[scale]["overall"]["R2"])
    #     mlflow.log_metric(f"{scale}_overall_abs_score", metrics[scale]["overall"]["abs_score"])

    #     for tgt, vals in metrics[scale]["per_target"].items():
    #         mlflow.log_metric(f"{scale}_{tgt}_mape", vals["MAPE"])
    #         mlflow.log_metric(f"{scale}_{tgt}_r2", vals["R2"])
    #         mlflow.log_metric(f"{scale}_{tgt}_abs_score", vals["abs_score"])

    for scale in ["scaled", "unscaled"]:
        mlflow.log_metric(f"eval/{scale}/overall/mape", metrics[scale]["overall"]["MAPE"])
        mlflow.log_metric(f"eval/{scale}/overall/r2", metrics[scale]["overall"]["R2"])
        mlflow.log_metric(f"eval/{scale}/overall/abs_score", metrics[scale]["overall"]["abs_score"])

        for tgt, vals in metrics[scale]["per_target"].items():
            mlflow.log_metric(f"eval/{scale}/{tgt}/mape", vals["MAPE"])
            mlflow.log_metric(f"eval/{scale}/{tgt}/r2", vals["R2"])
            mlflow.log_metric(f"eval/{scale}/{tgt}/abs_score", vals["abs_score"])


    # artifacts: metrics.json + plots
    for p in eval_out["paths"].values():
        mlflow.log_artifact(p)

    # artifacts: checkpoint + best weights
    latest_ckpt = [f for f in os.listdir(output_dir) if f.endswith(".latest.pt")][0]
    latest_ckpt_path = os.path.join(output_dir, latest_ckpt)
    best_weights_path = os.path.join(output_dir, f"{uuid_str}.best_weights.pt")

    mlflow.log_artifact(latest_ckpt_path)
    mlflow.log_artifact(best_weights_path)

    # histories (lists) from checkpoint: log as artifact + per-epoch metrics
    ckpt = torch.load(latest_ckpt_path, map_location="cpu")
    histories = {
        "train_losses": ckpt["train_losses"],
        "val_losses": ckpt["val_losses"],
        "val_mapes": ckpt["val_mapes"],
        "val_r2s": ckpt["val_r2s"],
        "val_abs_scores": ckpt["val_abs_scores"],
    }

    hist_path = os.path.join(output_dir, "histories.json")
    with open(hist_path, "w") as f:
        json.dump(histories, f, indent=2)
    mlflow.log_artifact(hist_path)

    for i in range(len(histories["train_losses"])):
        mlflow.log_metric("history/train_loss", float(histories["train_losses"][i]), step=i)
        mlflow.log_metric("history/val_loss", float(histories["val_losses"][i]), step=i)
        mlflow.log_metric("history/val_mape", float(histories["val_mapes"][i]), step=i)
        mlflow.log_metric("history/val_r2", float(histories["val_r2s"][i]), step=i)
        mlflow.log_metric("history/val_abs_score", float(histories["val_abs_scores"][i]), step=i)

    # log best model (model already has best weights loaded)
    if input_example is not None:
        x_example = input_example.detach().cpu()
        was_training = model.training
        model.eval()
        with torch.no_grad():
            y_example = model(x_example.to(next(model.parameters()).device)).detach().cpu()
        if was_training:
            model.train()
        signature = infer_signature(x_example.numpy(), y_example.numpy())
    else:
        signature = None


    mlflow.pytorch.log_model(
        model,
        name="model_best",
        input_example=x_example.numpy() if input_example is not None else None,
        signature=signature
    )